# Part 3 — Skills and the critic loop

Two ideas, both about **quality**:

| | Problem | Fix |
|---|---|---|
| **Skills** | Your system prompt keeps growing. Detailed formatting rules bloat every call, even when irrelevant. | Put them in a `SKILL.md` the agent loads *only when it needs it*. |
| **Critic loop** | The agent grades its own homework. Nothing checks whether claims are actually sourced. | A separate `fact-checker` subagent with fresh context reviews the report and sends it back. |

The critic loop is a **generator–evaluator** pattern: one agent produces, a second judges it against
explicit criteria, and the work loops until it passes or hits a round cap. Building it from subagents
rather than hand-wired graph nodes means the evaluator gets its own clean context and its own tools
for free.

In [ ]:
from dotenv import load_dotenv

load_dotenv(override=True)

import deepagents
from deepagents import create_deep_agent
from deepagents.backends import FilesystemBackend
from IPython.display import Markdown, display

from research_tools import ensure_sandbox, internet_search, run_agent, show_tools, show_tree

SANDBOX = ensure_sandbox()
MODEL = "anthropic:claude-sonnet-5"

print(f"deepagents {deepagents.__version__}  |  {MODEL}")
print(f"sandbox: {SANDBOX}")

---
## Skills — progressive disclosure

A skill is just **a folder with a `SKILL.md` inside**. Nothing more.

```
sandbox/skills/
├── analyst-report/
│   └── SKILL.md
└── swot-analysis/
    └── SKILL.md
```

**Progressive disclosure** is the whole point:

1. At startup the agent sees only each skill's `name` + `description` — a couple of lines each.
2. When it judges a skill relevant, it **reads the full file itself** with `read_file`.

So a 2,000-word style guide costs you ~15 tokens of context until the moment it's actually needed.
That's how you can have twenty skills without drowning the context window.

### `SKILL.md` anatomy

```markdown
---
name: analyst-report          <- MUST match the folder name exactly
description: When to use this, and when NOT to.
---

# Everything else is normal markdown instructions
```

⚠️ **Two things that silently break skills:**
- `name` not matching the parent folder → the skill is skipped.
- Missing or malformed YAML frontmatter → skipped, with only a log warning.

Both fail quietly. Verify skills loaded — the next cell does exactly that.

The `description` is doing prompt-engineering work: it is the *only* thing the agent sees when
deciding whether to open the file. Say when to use it **and when not to**.

In [ ]:
# What's actually on disk, and what the agent will see at startup.
import re

for skill_md in sorted((SANDBOX / "skills").glob("*/SKILL.md")):
    text = skill_md.read_text(encoding="utf-8")
    fm = re.match(r"^---\s*\n(.*?)\n---\s*\n", text, re.DOTALL)
    folder = skill_md.parent.name

    if not fm:
        print(f"[BROKEN] {folder}: no YAML frontmatter -- this skill will be silently skipped")
        continue

    name = re.search(r"^name:\s*(.+)$", fm.group(1), re.M).group(1).strip()
    desc = re.search(r"^description:\s*(.+)$", fm.group(1), re.M).group(1).strip()
    ok = "OK" if name == folder else "MISMATCH -> will be skipped!"

    print(f"[{ok}] folder={folder!r} name={name!r}")
    print(f"   description ({len(desc)} chars, this is ALL the agent sees up front):")
    print(f"   {desc}")
    print(f"   full file is {len(text):,} chars -- loaded only on demand\n")

### Wiring skills in

One argument: `skills=["/skills/"]`. That's a **virtual path** — it resolves inside the sandbox,
same as everything else. You can pass several sources; later ones override earlier ones when names
collide (useful for `user` skills overriding `project` defaults).

In [ ]:
skilled_agent = create_deep_agent(
    model=MODEL,
    tools=[internet_search],
    system_prompt=(
        "You are a competitive intelligence analyst.\n"
        "You have SKILLS available. Before writing any report or SWOT, read the relevant "
        "SKILL.md in full and follow it exactly. Do not improvise your own format."
    ),
    backend=FilesystemBackend(root_dir=SANDBOX, virtual_mode=True),
    skills=["/skills/"],
)

show_tools(skilled_agent, "skilled_agent")

In [ ]:
# Ask for a SWOT and watch it go READ the skill before writing.
# The READ line is progressive disclosure happening in front of you.
result = run_agent(
    skilled_agent,
    "Produce a SWOT analysis of Zepto. Research it first, then write to /output/zepto_swot.md.",
)

In [ ]:
swot = SANDBOX / "output" / "zepto_swot.md"
display(Markdown(swot.read_text(encoding="utf-8"))) if swot.exists() else show_tree()

**Check the output against the skill.** Did it produce the `### The strategic tension` section? Did
entries use the `claim — evidence [n] → consequence` format? If yes, the skill did its job — none of
that was in the system prompt.

> **Experiment worth doing:** delete `skills=["/skills/"]` from the cell above, re-run, and compare.
> You'll get a generic four-list SWOT. That difference *is* the feature.

---
## The critic loop

Here's the uncomfortable truth about everything we've built so far: **nothing verifies it.** The
agent writes "revenue grew 40% [3]" and we take its word that source 3 says that.

Self-review doesn't fix this. An agent asked to check its own work is reading its own context and
reasoning — it already believes the claim. What works is a **fresh, adversarial context**:

```
        lead
         |
    +----+---------------------+
    |                          |
  report-writer            fact-checker
  (writes /output/..)      (reads it cold, tries to BREAK it)
         ^                     |
         |     FAIL + issues   |
         +---------------------+
              revise, re-check (max 2 rounds)
```

Two design choices that matter:

1. **The fact-checker gets `internet_search`.** It doesn't just check that a URL is *present*, it
   checks the URL actually *supports the claim*. Without search it's a formatting linter.
2. **It's told to default to FAIL when unsure.** A critic that wants to approve is worthless — the
   evaluator needs explicit, checkable success criteria and a bias toward rejection.

In [ ]:
competitor_researcher = {
    "name": "competitor-researcher",
    "description": (
        "Researches ONE company in depth and writes /research/<company>.md. "
        "Delegate one call per company -- never two companies in one call. "
        "Returns a short summary plus the file path."
    ),
    "system_prompt": """You research exactly ONE company, thoroughly.

Run at least 4 internet_search calls from different angles: business model,
funding/financials, market position, and risks.

Write to /research/<company_lowercase>.md with sections:
  # <Company> / ## Business model / ## Financials & funding / ## Market position / ## Risks / ## Sources

Put the source URL inline next to every single claim. If you could not confirm
something, write [unverified] -- never guess a number.

FINAL REPLY: 5-8 bullets of key findings plus the file path. Keep it short --
the detail belongs in the file, not in your reply.""",
    "model": MODEL,
}

report_writer = {
    "name": "report-writer",
    "description": (
        "Writes the final client-facing report from existing /research/ files, following the "
        "analyst-report house style. Use AFTER all research is done. Can also revise an existing "
        "report when given fact-checker issues to fix."
    ),
    "system_prompt": """You write the final report.

1. FIRST read /skills/analyst-report/SKILL.md in full and follow it exactly.
   Its structure is mandatory -- do not invent your own sections.
2. Read every file in /research/ with read_file. Those are your only facts.
3. Write the report to the path you were given.

Never introduce a fact that is not in the research files. If the research says
[unverified], your report says [unverified] too -- do not upgrade uncertainty
into confidence.

If you were given fact-checker issues, fix EVERY one and say what you changed.""",
    "model": MODEL,
}

fact_checker = {
    "name": "fact-checker",
    "description": (
        "Adversarially verifies a finished report: checks every claim is sourced and that sources "
        "actually support the claims. Returns PASS or FAIL with a numbered issue list. "
        "Use after report-writer, before showing anything to the user."
    ),
    "system_prompt": """You are an adversarial fact-checker. Your job is to BREAK the report,
not to approve it. A critic who wants to approve is useless.

Given a report path:
1. read_file the report.
2. read_file the /research/ files it was built from.
3. Extract every quantitative or factual claim.
4. For each one check:
   - Is there a source URL?
   - Does the research file actually support it, or was it embellished?
   - Is inference being presented as fact?
   - Was an [unverified] item quietly upgraded to a confident claim?
5. Use internet_search to independently spot-check the 3 most load-bearing numbers.
   Checking that a URL merely EXISTS is not fact-checking.
6. Confirm the report has all 7 required sections from the analyst-report skill,
   including a non-empty "Confidence & Limitations".

Write your findings to /research/factcheck_<n>.md.

FINAL REPLY format -- exactly this:

VERDICT: PASS   (or)   VERDICT: FAIL
ISSUES:
1. <section> - <what is wrong> - <what would fix it>
2. ...

Default to FAIL when you are unsure. Only PASS when every claim is sourced and
supported, and the structure is complete.""",
    "tools": [internet_search],
    "model": MODEL,
}

print("3 subagents defined:", competitor_researcher["name"], "|", report_writer["name"], "|", fact_checker["name"])

### The lead orchestrates the loop

Note the explicit **round cap** in the prompt. Without one, a strict critic and a stubborn writer can
ping-pong until you hit the recursion limit and burn a lot of money. Always bound your loops.

In [ ]:
PIPELINE_PROMPT = """You are the lead analyst. You COORDINATE; you do not research or write.

## Pipeline -- follow exactly
1. write_todos with your plan.
2. RESEARCH: one `task` to competitor-researcher per company. Send them in the
   same turn so they run concurrently. Do NOT call internet_search yourself.
3. WRITE: one `task` to report-writer, telling it the output path.
4. VERIFY: one `task` to fact-checker with the report path.
5. If the verdict is FAIL: send the issues back to report-writer to fix, then
   re-run fact-checker.
   HARD LIMIT: at most 2 revision rounds. After that, accept the report and record
   the unresolved issues in its "Confidence & Limitations" section.
6. Report to the user: the final path, the verdict, and how many rounds it took.

Keep your own replies short. The detail lives in the files.
"""

pipeline = create_deep_agent(
    model=MODEL,
    tools=[internet_search],
    system_prompt=PIPELINE_PROMPT,
    subagents=[competitor_researcher, report_writer, fact_checker],
    backend=FilesystemBackend(root_dir=SANDBOX, virtual_mode=True),
    skills=["/skills/"],
)

show_tools(pipeline, "pipeline")
print("\nsubagents: competitor-researcher, report-writer, fact-checker")

### Run the full thing

⏱️ **This is the expensive one — 5–15 minutes and a few dollars on Sonnet.** It runs 3 researchers,
a writer, a fact-checker, and possibly revision rounds.

To rehearse cheaply first: set `MODEL = "anthropic:claude-haiku-4-5"` at the top and re-run from
there. The output is worse but the machinery is identical.

Watch the `DELEGATE ->` lines. That trail *is* your architecture diagram.

In [ ]:
result = run_agent(
    pipeline,
    "Produce a competitive intelligence report on the Indian quick-commerce market, "
    "covering Zepto, Blinkit, and Swiggy Instamart. "
    "Write it to /output/quick_commerce_report.md.",
    recursion_limit=250,
)

print("\n" + "=" * 70)
print(result["messages"][-1].content)

In [ ]:
print("sandbox/")
show_tree()

In [ ]:
# The fact-checker's own working notes -- often more interesting than the report.
for fc in sorted((SANDBOX / "research").glob("factcheck*.md")):
    print(f"--- {fc.name} ---")
    print(fc.read_text(encoding="utf-8")[:1800])
    print()

In [ ]:
report = SANDBOX / "output" / "quick_commerce_report.md"
display(Markdown(report.read_text(encoding="utf-8"))) if report.exists() else show_tree()

---
## What you just learned

1. **Skills = progressive disclosure.** Name + description always in context; the full file loaded
   on demand. Twenty skills cost almost nothing until used.
2. **The `description` is the whole interface.** It's the only thing the agent reads when deciding.
   Say when to use it *and when not to*.
3. **Skills fail silently.** Wrong `name`, missing frontmatter → skipped with a log warning you
   won't see. Verify.
4. **A critic needs a fresh context and its own tools.** Self-review doesn't work; the agent already
   believes its own claims. Give the critic search and tell it to default to FAIL.
5. **Always bound your loops.** "At most 2 revision rounds" is the difference between a pipeline and
   a runaway bill.

### Try it yourself
- Write a third skill — `financial-modelling` — and see the agent pick it up with zero code changes.
  *That* is the payoff: new capability, no redeploy.
- Break a skill on purpose (rename the folder but not the `name:`) and watch it vanish silently.
- Give `fact-checker` a cheap model and the writer an expensive one. Does quality hold?
- Remove "default to FAIL when unsure" and count how many reports suddenly PASS first time.

### Next
**Part 4** — wrap this pipeline in a Streamlit UI so it's demoable, plus the README.